In [9]:
import os

PROCESSED_DIR = r'C:\Users\Shever\jupyter\ruda\processed_data'
masks_dir = os.path.join(PROCESSED_DIR, 'test', 'masks_talc')
images_dir = os.path.join(PROCESSED_DIR, 'test', 'otalkovannye')

masks = sorted([f for f in os.listdir(masks_dir) if f.endswith('.png')])
images = sorted([f for f in os.listdir(images_dir) if f.lower().endswith(('.jpg', '.jpeg', '.bmp'))])

# Создаём словарь изображений: base_name → полное имя
images_dict = {}
for img in images:
    base = os.path.splitext(img)[0]
    images_dict[base] = img

print("🔧 Исправление пар маска-изображение:\n")

fixed = 0
for mask_name in masks:
    mask_base = os.path.splitext(mask_name)[0]
    
    # Если точное имя нашлось — отлично
    if mask_base in images_dict:
        continue
    
    # Ищем ближайшее совпадение
    best_match = None
    for img_base, img_name in images_dict.items():
        # Проверяем по первым 10 символам + длине
        if len(mask_base) >= 10 and len(img_base) >= 10:
            if mask_base[:10] == img_base[:10] and len(mask_base) == len(img_base):
                best_match = img_name
                break
        # Для DSCN сравниваем полные имена (убираем расширение)
        if mask_base.startswith('DSCN') and img_base.startswith('DSCN'):
            if mask_base == img_base:
                best_match = img_name
                break
    
    if best_match:
        old_path = os.path.join(masks_dir, mask_name)
        new_name = os.path.splitext(best_match)[0] + '.png'
        new_path = os.path.join(masks_dir, new_name)
        
        if old_path != new_path and not os.path.exists(new_path):
            os.rename(old_path, new_path)
            print(f"  ✅ {mask_name} → {new_name}")
            fixed += 1

# Перепроверяем
masks = [f for f in os.listdir(masks_dir) if f.endswith('.png')]
images_set = set(os.path.splitext(img)[0] for img in images)

paired = 0
unpaired = []
for mask_name in masks:
    mask_base = os.path.splitext(mask_name)[0]
    if mask_base in images_set:
        paired += 1
    else:
        unpaired.append(mask_name)

print(f"\n✅ Исправлено: {fixed} масок")
print(f"✅ Итог: {paired}/{len(masks)} пар")

if unpaired:
    print(f"\n❌ Всё ещё без пары ({len(unpaired)}):")
    for m in unpaired:
        print(f"  - {m}")
        
        # Последний шанс — сравним по номеру образца
        for img_base in images_set:
            # Извлекаем номер (например 2550374 из 2550374-2)
            mask_parts = m.replace('.png', '').split('-')[0].split(' ')[0]
            img_parts = img_base.split('-')[0].split(' ')[0]
            if mask_parts == img_parts and len(mask_parts) >= 6:
                new_name = img_base + '.png'
                old_path = os.path.join(masks_dir, m)
                new_path = os.path.join(masks_dir, new_name)
                os.rename(old_path, new_path)
                print(f"    ✅ Исправлено по номеру: {m} → {new_name}")
                break

🔧 Исправление пар маска-изображение:

  ✅ 2550374-2 10x.png → 2550374-2 10х.png
  ✅ 2550375-1 10x.png → 2550375-1 10х.png
  ✅ 2550375-3 5x.png → 2550375-3 5х.png

✅ Исправлено: 3 масок
✅ Итог: 42/42 пар


In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
print("✅ Фикс OpenMP применён")

✅ Фикс OpenMP применён


In [3]:
# ============================================
# ШАГ 3.15: АНСАМБЛЬ U-Net B5 (768px + FocalLoss)
# ============================================

import torch
torch.backends.cudnn.benchmark = True

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import segmentation_models_pytorch as smp
import numpy as np
import os
from PIL import Image, ImageOps
import warnings
import logging

warnings.filterwarnings('ignore')
logging.getLogger('PIL').setLevel(logging.WARNING)

# ============================================
# КОНСТАНТЫ
# ============================================

IMAGE_SIZE = 768
PROCESSED_DIR = r'C:\Users\Shever\jupyter\ruda\processed_data'
MODEL_DIR = r'C:\Users\Shever\jupyter\ruda\models'
NUM_MODELS = 3

# ============================================
# DATASET
# ============================================

class TalcDataset(Dataset):
    def __init__(self, root_dir):
        self.images_dir = os.path.join(root_dir, 'test', 'otalkovannye')
        self.masks_dir = os.path.join(root_dir, 'test', 'masks_talc')
        self.images, self.masks = [], []
        for f in os.listdir(self.masks_dir):
            if not f.endswith('.png'): continue
            base = f.replace('.png', '')
            for ext in ['.JPG', '.jpg', '.jpeg', '.bmp']:
                p = os.path.join(self.images_dir, base + ext)
                if os.path.exists(p):
                    self.images.append(p)
                    self.masks.append(os.path.join(self.masks_dir, f))
                    break
        
        self.img_t = transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
        self.mask_t = transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.ToTensor(),
        ])
    
    def __len__(self): return len(self.images)
    
    def __getitem__(self, idx):
        img = ImageOps.autocontrast(Image.open(self.images[idx]).convert('RGB'), cutoff=2)
        mask = Image.open(self.masks[idx]).convert('L')
        return self.img_t(img), (self.mask_t(mask) > 0.5).float()

# ============================================
# МЕТРИКИ
# ============================================

def iou_score(pred, true, eps=1e-6):
    pred, true = pred.view(-1), true.view(-1)
    inter = (pred * true).sum()
    return (inter + eps) / (pred.sum() + true.sum() - inter + eps)

def dice_coef(pred, true, eps=1e-6):
    pred, true = pred.view(-1), true.view(-1)
    inter = (pred * true).sum()
    return (2*inter + eps) / (pred.sum() + true.sum() + eps)

def talc_error(pred, true):
    return abs(true.sum().item()/true.numel()*100 - pred.sum().item()/pred.numel()*100)

# ============================================
# ОБУЧЕНИЕ ОДНОЙ МОДЕЛИ
# ============================================

def train_one_model(model, train_loader, val_loader, device, model_idx, epochs_stage1=10, epochs_stage2=20):
    
    class CombinedLoss(nn.Module):
        def __init__(self):
            super().__init__()
            self.dice = smp.losses.DiceLoss(mode='binary')
            self.focal = smp.losses.FocalLoss(mode='binary')
        def forward(self, p, t): return self.dice(p, t) + self.focal(p, t)
    
    criterion = CombinedLoss()
    
    # ЭТАП 1
    for param in model.encoder.parameters():
        param.requires_grad = False
    
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=3e-4, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs_stage1, eta_min=1e-6)
    
    best_val_err = float('inf')
    best_state = None
    
    print(f"\n  ┌─ Этап 1: Заморожен энкодер ─────────────────────┐")
    
    for epoch in range(epochs_stage1):
        model.train()
        train_iou = train_err = 0
        for img, msk in train_loader:
            img, msk = img.to(device), msk.to(device)
            optimizer.zero_grad()
            out = model(img)
            loss = criterion(out, msk)
            loss.backward()
            optimizer.step()
            pred = (torch.sigmoid(out) > 0.5).float()
            train_iou += iou_score(pred, msk).item()
            train_err += talc_error(pred, msk)
        
        model.eval()
        val_iou = val_err = val_dice = 0
        with torch.no_grad():
            for img, msk in val_loader:
                img, msk = img.to(device), msk.to(device)
                pred = (torch.sigmoid(model(img)) > 0.5).float()
                val_iou += iou_score(pred, msk).item()
                val_dice += dice_coef(pred, msk).item()
                val_err += talc_error(pred, msk)
        
        n = len(train_loader)
        train_iou /= n; train_err /= n
        val_iou /= len(val_loader); val_dice /= len(val_loader); val_err /= len(val_loader)
        
        scheduler.step()
        
        status = ""
        if val_err < best_val_err:
            best_val_err = val_err
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            status = "💾"
        
        print(f"  │ Epoch {epoch+1:2d}/{epochs_stage1} | Tr IoU: {train_iou:.4f} | Val IoU: {val_iou:.4f} | Val Dice: {val_dice:.4f} | Val Err: {val_err:.2f}% | lr: {scheduler.get_last_lr()[0]:.2e} {status}")
    
    model.load_state_dict(best_state)
    print(f"  └──────────────────────────────────────────────────┘")
    print(f"    ✅ Этап 1: Best Val Err: {best_val_err:.2f}%")
    
    # ЭТАП 2
    for param in model.parameters():
        param.requires_grad = True
    
    optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs_stage2, eta_min=1e-7)
    
    best_val_err = float('inf')
    
    print(f"\n  ┌─ Этап 2: Разморожен весь ────────────────────────┐")
    
    for epoch in range(epochs_stage2):
        model.train()
        train_iou = train_err = 0
        for img, msk in train_loader:
            img, msk = img.to(device), msk.to(device)
            optimizer.zero_grad()
            out = model(img)
            loss = criterion(out, msk)
            loss.backward()
            optimizer.step()
            pred = (torch.sigmoid(out) > 0.5).float()
            train_iou += iou_score(pred, msk).item()
            train_err += talc_error(pred, msk)
        
        model.eval()
        val_iou = val_err = val_dice = 0
        with torch.no_grad():
            for img, msk in val_loader:
                img, msk = img.to(device), msk.to(device)
                pred = (torch.sigmoid(model(img)) > 0.5).float()
                val_iou += iou_score(pred, msk).item()
                val_dice += dice_coef(pred, msk).item()
                val_err += talc_error(pred, msk)
        
        n = len(train_loader)
        train_iou /= n; train_err /= n
        val_iou /= len(val_loader); val_dice /= len(val_loader); val_err /= len(val_loader)
        
        scheduler.step()
        
        status = ""
        if val_err < best_val_err:
            best_val_err = val_err
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            status = "💾"
        
        print(f"  │ Epoch {epoch+1:2d}/{epochs_stage2} | Tr IoU: {train_iou:.4f} | Val IoU: {val_iou:.4f} | Val Dice: {val_dice:.4f} | Val Err: {val_err:.2f}% | lr: {scheduler.get_last_lr()[0]:.2e} {status}")
    
    model.load_state_dict(best_state)
    print(f"  └──────────────────────────────────────────────────┘")
    print(f"    ✅ Этап 2: Best Val Err: {best_val_err:.2f}%")
    
    return model, best_val_err

# ============================================
# АНСАМБЛЬ
# ============================================

def ensemble_predict(models, image, device):
    preds = []
    for model in models:
        model.eval()
        with torch.no_grad():
            pred = torch.sigmoid(model(image.unsqueeze(0).to(device)))
            preds.append(pred)
    avg_pred = torch.stack(preds).mean(dim=0)
    return (avg_pred > 0.5).float()

# ============================================
# MAIN
# ============================================

if __name__ == '__main__':
    
    os.makedirs(MODEL_DIR, exist_ok=True)
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    BATCH_SIZE = 2
    NUM_WORKERS = 0
    
    print(f"╔══════════════════════════════════════════╗")
    print(f"║  АНСАМБЛЬ {NUM_MODELS}× U-Net B5 | 768px | FocalLoss ║")
    print(f"╚══════════════════════════════════════════╝")
    print(f"🖥️  {DEVICE}")
    
    full_dataset = TalcDataset(PROCESSED_DIR)
    
    train_loader = DataLoader(full_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader = DataLoader(full_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    
    print(f"\n📊 Train: {len(full_dataset)} примеров")
    
    models = []
    errors = []
    
    for i in range(NUM_MODELS):
        print(f"\n{'='*60}")
        print(f"  🧠 МОДЕЛЬ {i+1}/{NUM_MODELS}")
        print(f"{'='*60}")
        
        model = smp.Unet(
            encoder_name='efficientnet-b5',
            encoder_weights='imagenet',
            in_channels=3,
            classes=1,
        ).to(DEVICE)
        
        model, best_err = train_one_model(model, train_loader, val_loader, DEVICE, i)
        models.append(model)
        errors.append(best_err)
        
        torch.save(model.state_dict(), os.path.join(MODEL_DIR, f'unet_b5_768px_v2_{i+1}.pth'))
        print(f"  ✅ Модель {i+1} сохранена. Best Val Err: {best_err:.2f}%")
    
    print(f"\n{'='*70}")
    print(f"АНСАМБЛЬ ГОТОВ!")
    print(f"{'='*70}")
    print(f"Ошибки моделей: {[f'{e:.2f}%' for e in errors]}")
    print(f"Средняя ошибка: {np.mean(errors):.2f}%")
    
    print(f"\n📊 Проверка ансамбля...")
    dataset = TalcDataset(PROCESSED_DIR)
    total_err = total_iou = 0
    for i in range(len(dataset)):
        img, true_mask = dataset[i]
        pred_mask = ensemble_predict(models, img, DEVICE)
        total_err += talc_error(pred_mask, true_mask)
        total_iou += iou_score(pred_mask, true_mask).item()
    
    print(f"  Ансамбль IoU: {total_iou/len(dataset):.4f}")
    print(f"  Ансамбль Err: {total_err/len(dataset):.2f}%")
    print(f"\n✅ Все модели сохранены в {MODEL_DIR}/")

╔══════════════════════════════════════════╗
║  АНСАМБЛЬ 3× U-Net B5 | 768px | FocalLoss ║
╚══════════════════════════════════════════╝
🖥️  cuda

📊 Train: 42 примеров

  🧠 МОДЕЛЬ 1/3

  ┌─ Этап 1: Заморожен энкодер ─────────────────────┐
  │ Epoch  1/10 | Tr IoU: 0.0849 | Val IoU: 0.0796 | Val Dice: 0.1453 | Val Err: 17.42% | lr: 2.93e-04 💾
  │ Epoch  2/10 | Tr IoU: 0.1275 | Val IoU: 0.1403 | Val Dice: 0.2386 | Val Err: 33.01% | lr: 2.71e-04 
  │ Epoch  3/10 | Tr IoU: 0.1937 | Val IoU: 0.1838 | Val Dice: 0.3013 | Val Err: 19.73% | lr: 2.38e-04 
  │ Epoch  4/10 | Tr IoU: 0.2327 | Val IoU: 0.2275 | Val Dice: 0.3561 | Val Err: 4.29% | lr: 1.97e-04 💾
  │ Epoch  5/10 | Tr IoU: 0.2662 | Val IoU: 0.2563 | Val Dice: 0.3901 | Val Err: 5.89% | lr: 1.50e-04 
  │ Epoch  6/10 | Tr IoU: 0.2850 | Val IoU: 0.2579 | Val Dice: 0.3945 | Val Err: 9.44% | lr: 1.04e-04 
  │ Epoch  7/10 | Tr IoU: 0.3254 | Val IoU: 0.3164 | Val Dice: 0.4578 | Val Err: 3.55% | lr: 6.26e-05 💾
  │ Epoch  8/10 | Tr IoU: 0.3457 | 

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

In [5]:
# Замени функцию ensemble_predict:
def ensemble_predict(models, image, device):
    preds = []
    for model in models:
        model.eval()
        with torch.no_grad():
            pred = torch.sigmoid(model(image.unsqueeze(0).to(device)))
            preds.append(pred.cpu())  # ← переносим на CPU
    avg_pred = torch.stack(preds).mean(dim=0)
    return (avg_pred > 0.5).float()

# Замени блок проверки ансамбля:
print(f"\n📊 Проверка ансамбля на всех 42 примерах...")

dataset = TalcDataset(PROCESSED_DIR)
loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)

total_err = 0
total_iou = 0

for img, true_mask in loader:
    pred_mask = ensemble_predict(models, img.squeeze(0), DEVICE)
    total_err += talc_error(pred_mask, true_mask)
    total_iou += iou_score(pred_mask, true_mask).item()

print(f"  Ансамбль IoU: {total_iou/len(dataset):.4f}")
print(f"  Ансамбль Err: {total_err/len(dataset):.2f}%")


📊 Проверка ансамбля на всех 42 примерах...
  Ансамбль IoU: 0.4058
  Ансамбль Err: 1.30%
